<a href="https://colab.research.google.com/github/RoshanHelmy/FlyRank-Assignment1-/blob/main/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*


I will use a Decision Tree Classifier for this lane.

The goal is to identify pages that may need content review based on observable signals such as impressions, CTR, average search position, content age, and word count.

A Decision Tree fits this problem because it can combine several signals into readable if/else decisions. It is also easier to interpret than a more complex model, which is useful because this project is intended to provide decision support rather than maximize model complexity.

I will compare the model against the Week-4 hand-written baseline using the same evaluation metric.

In [1]:
import os
import gc
import numpy as np
import pandas as pd

from huggingface_hub import hf_hub_download

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
from sklearn.inspection import permutation_importance

print("Libraries loaded successfully.")

Libraries loaded successfully.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## 2. Split design

I will use a grouped train/test split by client.

Pages from the same client should not appear in both training and testing because pages belonging to one client can have similar characteristics. Keeping clients separated gives a more honest estimate of how the model may perform on unseen clients.

The target is the declining proxy created from the available performance signals. I will use only features that would be observable at the decision time and exclude label-derived or future-window information.

In [3]:
from sklearn.model_selection import GroupShuffleSplit

features = [
    "impressions",
    "ctr",
    "avg_position",
    "content_age_days",
    "word_count"
]

target = "declining_proxy"
group_col = "client_hash_id"

model_data = model_df[
    features + [target, group_col, "content_hash_id"]
].copy()

# Clean numeric values
for col in features:
    model_data[col] = pd.to_numeric(
        model_data[col], errors="coerce"
    ).fillna(0)

model_data[target] = model_data[target].astype(int)

# Grouped split by client
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        model_data[features],
        model_data[target],
        groups=model_data[group_col]
    )
)

train = model_data.iloc[train_idx].copy()
test = model_data.iloc[test_idx].copy()

print("Train rows:", len(train))
print("Test rows:", len(test))
print("Train clients:", train[group_col].nunique())
print("Test clients:", test[group_col].nunique())

print(
    "Clients overlap:",
    len(
        set(train[group_col])
        & set(test[group_col])
    )
)

Train rows: 138310
Test rows: 38428
Train clients: 37
Test clients: 10
Clients overlap: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*


I will train a depth-3 Decision Tree using the five observable features.

The model will produce a probability score for the declining proxy. Pages can then be ranked by this score.

For a fair comparison, the model and the baseline will be evaluated on the same held-out test set using Precision@50. This focuses on the quality of the highest-priority recommendations rather than overall accuracy.

In [4]:
from sklearn.tree import DecisionTreeClassifier

X_train = train[features]
y_train = train[target]

X_test = test[features]
y_test = test[target]

tree = DecisionTreeClassifier(
    max_depth=3,
    class_weight="balanced",
    random_state=42
)

tree.fit(X_train, y_train)

model_scores = tree.predict_proba(X_test)[:, 1]

print("Decision Tree trained successfully.")

Decision Tree trained successfully.


In [5]:
import numpy as np

def precision_at_k(scores, labels, k=50):
    scores = np.asarray(scores)
    labels = np.asarray(labels)

    order = np.argsort(-scores)
    top_k = labels[order[:k]]

    return top_k.mean()

model_precision_50 = precision_at_k(
    model_scores,
    y_test,
    50
)

print(
    f"Decision Tree Precision@50: "
    f"{model_precision_50:.3f}"
)

Decision Tree Precision@50: 1.000


In [6]:
test["baseline_score"] = (
    (test["content_age_days"] >= 180).astype(int)
    * (test["impressions"] >= 500).astype(int)
    * test["impressions"]
)

baseline_precision_50 = precision_at_k(
    test["baseline_score"],
    test[target],
    50
)

print(
    f"Baseline Precision@50: "
    f"{baseline_precision_50:.3f}"
)

print(
    f"Decision Tree Precision@50: "
    f"{model_precision_50:.3f}"
)

Baseline Precision@50: 0.040
Decision Tree Precision@50: 1.000


In [7]:
comparison = pd.DataFrame({
    "Method": [
        "Week-4 Baseline",
        "Decision Tree"
    ],
    "Precision@50": [
        baseline_precision_50,
        model_precision_50
    ]
})

comparison

,Method,Precision@50
0,Week-4 Baseline,0.04
1,Decision Tree,1.00


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The model should be interpreted as decision support rather than as a causal explanation.

I will inspect the tree rules and the highest-ranked predictions to understand where the model succeeds and where it makes questionable recommendations.

A prediction can be wrong because the available signals do not contain enough information to identify whether a page truly needs refreshing. For example, a page can have low impressions or an old content age while still being intentionally maintained and useful.

In [8]:
from sklearn.tree import export_text

print(
    export_text(
        tree,
        feature_names=features
    )
)

|--- avg_position <= 15.00
|   |--- class: 0
|--- avg_position >  15.00
|   |--- impressions <= 99.50
|   |   |--- class: 0
|   |--- impressions >  99.50
|   |   |--- avg_position <= 15.00
|   |   |   |--- class: 1
|   |   |--- avg_position >  15.00
|   |   |   |--- class: 1



In [9]:
review = test[
    [
        "content_hash_id",
        "impressions",
        "ctr",
        "avg_position",
        "content_age_days",
        "word_count",
        target
    ]
].copy()

review["model_score"] = model_scores

top10 = review.sort_values(
    "model_score",
    ascending=False
).head(10)

top10

,content_hash_id,impressions,ctr,avg_position,content_age_days,word_count,declining_proxy,model_score
104492,content_0ecfd07cb87a5c4c,430,0.000000,16.512805,34,0.0,1,1.0
122368,content_b5a159e9a5e55e31,277,0.000000,47.441480,-50,2760.0,1,1.0
129993,content_fc4d094d350e1fce,426,0.000000,24.184240,-48,0.0,1,1.0
129994,content_fc4f5946c4a7e17e,434,0.002304,22.497950,34,0.0,1,1.0
104486,content_0ec07519f62eb1a5,693,0.000000,27.257619,34,0.0,1,1.0
129996,content_fc552cc94d1a68c0,721,0.000000,16.742480,34,0.0,1,1.0
113695,content_64b282e0453734e4,122,0.000000,19.789368,-48,0.0,1,1.0
113696,content_64b31377ab1d325a,195,0.000000,40.395377,-48,0.0,1,1.0
113699,content_64bf8c31b5cb0ec8,411,0.000000,15.978559,-48,0.0,1,1.0
104479,content_0eb50f9b9426c17f,191,0.000000,18.636990,-48,0.0,1,1.0


In [10]:
top50 = review.sort_values(
    "model_score",
    ascending=False
).head(50)

top50["correct"] = (
    top50[target] == 1
)

print(
    "Correct declining pages in top 50:",
    int(top50["correct"].sum())
)

print(
    "Non-declining pages in top 50:",
    int((~top50["correct"]).sum())
)

Correct declining pages in top 50: 50
Non-declining pages in top 50: 0


In [11]:
print("Top-50 model recommendations:")
print(
    top50[
        [
            "impressions",
            "ctr",
            "avg_position",
            "content_age_days",
            "word_count",
            target
        ]
    ].head(10)
)

Top-50 model recommendations:
        impressions       ctr  avg_position  content_age_days  word_count  \
104492          430  0.000000     16.512805                34         0.0   
122368          277  0.000000     47.441480               -50      2760.0   
129993          426  0.000000     24.184240               -48         0.0   
129994          434  0.002304     22.497950                34         0.0   
104486          693  0.000000     27.257619                34         0.0   
129996          721  0.000000     16.742480                34         0.0   
113695          122  0.000000     19.789368               -48         0.0   
113696          195  0.000000     40.395377               -48         0.0   
113699          411  0.000000     15.978559               -48         0.0   
104479          191  0.000000     18.636990               -48         0.0   

        declining_proxy  
104492                1  
122368                1  
129993                1  
129994            

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.